In [1]:
# Preprocessing Pipeline
# Goal: Clean clinical notes, prepare for NLP modeling
# preserve structure for active learning and fairness

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Libraries loaded")

Libraries loaded


In [2]:
# Load data and filter to top 10 specialties
df = pd.read_csv("../data/raw/mtsamples.csv", index_col=0)

top10 = df['medical_specialty'].value_counts().head(10).index.tolist()
df = df[df['medical_specialty'].isin(top10)].copy()

print("Shape after filtering:", df.shape)
print("\nSpecialties:", top10)

Shape after filtering: (3657, 5)

Specialties: [' Surgery', ' Consult - History and Phy.', ' Cardiovascular / Pulmonary', ' Orthopedic', ' Radiology', ' General Medicine', ' Gastroenterology', ' Neurology', ' SOAP / Chart / Progress Notes', ' Obstetrics / Gynecology']


In [3]:
# Drop rows with missing transcription text
df = df.dropna(subset=['transcription'])

# Clean text - lowercase, remove extra whitespace
df['clean_text'] = df['transcription'].str.lower().str.strip()
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True)

print("Shape after cleaning:", df.shape)
print("\nSample cleaned text:")
print(df['clean_text'].iloc[0][:300])

Shape after cleaning: (3630, 6)

Sample cleaned text:
2-d m-mode: , ,1. left atrial enlargement with left atrial diameter of 4.7 cm.,2. normal size right and left ventricle.,3. normal lv systolic function with left ventricular ejection fraction of 51%.,4. normal lv diastolic function.,5. no pericardial effusion.,6. normal morphology of aortic valve, mi


In [4]:
# Encode specialty labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['medical_specialty'])

print("Label mapping:")
for i, specialty in enumerate(le.classes_):
    print(f"  {i}: {specialty}")

Label mapping:
  0:  Cardiovascular / Pulmonary
  1:  Consult - History and Phy.
  2:  Gastroenterology
  3:  General Medicine
  4:  Neurology
  5:  Obstetrics / Gynecology
  6:  Orthopedic
  7:  Radiology
  8:  SOAP / Chart / Progress Notes
  9:  Surgery


In [5]:
# Truncate text to 512 characters (BERT limit later)
df['clean_text'] = df['clean_text'].str[:512]

# Split into train and test
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (2904,)
Test size: (726,)


In [6]:
# Save processed data
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

# Save label mapping
label_map = {i: specialty for i, specialty in enumerate(le.classes_)}
pd.DataFrame(label_map.items(), columns=['label', 'specialty']).to_csv(
    "../data/processed/label_map.csv", index=False)

print("All files saved successfully")

All files saved successfully
